# **Preprocessing, EDA, dan BERTopic**

Satu alur: `raw.csv` -> pecah kalimat -> pembersihan -> **dedup** -> EDA ringkas -> stopword -> **BERTopic + outlier handling**. Teks yang di-embed tetap natural (tanpa stemming/stopword/lowercase).

**Output hanya 2 file:**
1. `data/processed/sentences_with_topic.csv` - seluruh kalimat clean (tanpa duplikat) + kolom `topic`.
2. `data/processed/topic_info.csv` - ringkasan tiap topik + 3 kalimat representatif.

## **1. Import & baca data mentah**

In [ ]:
import re
import unicodedata
from pathlib import Path
from collections import Counter
import pandas as pd

raw = pd.read_csv("data/raw/raw.csv", dtype={"id_artikel": str}, keep_default_na=False)
required = {"id_artikel", "judul", "isi_mentah", "timestamp"}
if not required.issubset(raw.columns) or raw["id_artikel"].duplicated().any() or raw["id_artikel"].eq("").any():
    raise ValueError("Kolom wajib tidak lengkap, atau id_artikel kosong/duplikat.")
print(f"Artikel mentah: {len(raw):,} | isi kosong: {raw['isi_mentah'].str.strip().eq('').sum():,}")

## **2. Pecah kalimat + pembersihan + dedup (satu alur)**

Pecah tiap artikel jadi kalimat, bersihkan noise (dateline, URL, footer, disclaimer, listing saham/kurs, boilerplate, kalimat terlalu pendek/panjang), lalu **buang duplikat** sehingga kalimat yang berulang di banyak artikel (boilerplate seperti promo Prioritas) hanya disimpan satu kali. Kalimat yang dibuang tidak disimpan ke file terpisah.

In [ ]:
# --- pola & kamus ---
POLA_BANK = {
    "BCA":     r"\bbca\b|\bbbca\b|bank central asia",
    "BRI":     r"\bbri\b|\bbbri\b|bank rakyat indonesia|\bbrilink\b",
    "BNI":     r"\bbni\b|\bbbni\b|bank negara indonesia",
    "Mandiri": r"\bmandiri\b|\bbmri\b|bank mandiri",
}
SINGKATAN = ["PT", "Tbk", "Persero", "Rp", "US", "No", "Jl", "Dr", "Ir", "Hj",
             "Prof", "Sdr", "S.E", "M.M", "Ph.D", "dll", "dsb", "dkk", "a.l", "yakni"]

SPLIT      = re.compile(r'(?<=[.!?])\s+|(?<=[.!?]["\u201d])\s+')
DATELINE   = re.compile(r'^\s*(?:jakarta\s*,?\s*)?cnbc\s+indonesia\s*[-\u2013\u2014:]\s*', re.I)
URL        = re.compile(r'(?i)\b(?:https?://|www\.)[^\s<>"\u201d]+')
INVISIBLE  = re.compile('[\u200b\u200c\u200d\u200e\u200f\u2060\ufeff]')
FOOTER     = re.compile(r'^(?:cnbc\s+indonesia\s+research(?:\s+market@cnbcindonesia\.com)?|selamat\s+berinvestasi\s+secara\s+bijak)[.!\s]*$', re.I)
LINK_PROMO = re.compile(r'(?i)(?:baca\s+artikel\s+cnbc\s+indonesia|selengkapnya\s+di\s+sini).*(?:cnbcindonesia\.com|download\s+apps\s+cnbc)')
DISCLAIMER = re.compile(r"^keputusan berinvestasi sepenuhnya berada di tangan masing-masing investor sesuai dengan profil risiko dan tujuan keuangan pribadi[.!\s]*$", re.I)
FRAGMENT   = re.compile(r"^(?:ltd\.?|\(?dok\.?)$", re.I)
NUMERIC    = re.compile(r'^[\W\d_]+$', re.UNICODE)
SAMPAH     = re.compile(r"CNBC\s*INDONESIA\s*RESEARCH|market@cnbcindonesia\.com|\bNEXT\b|Lihat\s*juga|Saksikan\s+(?:Video|juga)|\bBaca\s*:|^Foto\s*:|^Infografis\s*:|\[Gambas:|\bBaca\s+(?:juga|artikel)\b", re.I)

# listing saham / tabel kurs
KODE_EMITEN = re.compile(r"\([A-Z]{4}\)")
RP_NOMINAL  = re.compile(r"Rp[\s.]?\d")
KURS_LABEL  = re.compile(r"(?i)\b(?:e-?rate|tt\s*counter|bank\s*notes|special\s*rates?|ttb|tts)\b|-?\s*(?:beli|jual)\s*:")

def is_listing_noise(text):
    if len(KURS_LABEL.findall(text)) >= 3:
        return True
    if len(KODE_EMITEN.findall(text)) >= 3 and len(RP_NOMINAL.findall(text)) >= 3:
        return True
    if "Rekomendasi Saham" in text or len(re.findall(r"\|", text)) >= 2 or re.search(r"\bTP\b.*\bSL\b", text):
        return True
    return False

def deteksi_bank(teks):
    return ", ".join(b for b, pat in POLA_BANK.items() if re.search(pat, teks, flags=re.I))

def pecah_kalimat(teks):
    t = re.sub(r"\s+", " ", str(teks)).strip()
    for ab in SINGKATAN:
        t = re.sub(rf"\b{re.escape(ab)}\.", lambda m: m.group(0)[:-1] + "<DOT>", t)
    t = re.sub(r"\b([A-Z])\.(?=\s)", r"\1<DOT>", t)
    t = re.sub(r"(?<=\d)\.(?=\d)", "<DOT>", t)
    return [x.replace("<DOT>", ".").strip() for x in SPLIT.split(t) if x.strip()]

def bersihkan(text):
    text = unicodedata.normalize("NFC", str(text))
    text = INVISIBLE.sub("", text)
    text = re.sub(r"\s+", " ", text).strip()
    if LINK_PROMO.search(text):
        return ""
    text = DATELINE.sub("", text)
    text = re.sub(r'^\s*[\u2013\u2014\u2022\u00b7-]+\s*', '', text)
    text = URL.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if not text or FOOTER.fullmatch(text) or DISCLAIMER.fullmatch(text) \
       or FRAGMENT.fullmatch(text) or NUMERIC.fullmatch(text) \
       or SAMPAH.search(text) or is_listing_noise(text):
        return ""
    return text

# --- jalankan: pecah + bersih ---
rows = []
for row in raw.itertuples(index=False):
    banks = deteksi_bank(f"{row.judul} {row.isi_mentah}")
    for i, asli in enumerate(pecah_kalimat(row.isi_mentah)):
        bersih = bersihkan(asli)
        if not bersih:
            continue
        n = len(bersih.split())
        if n < 4 or n > 80:          # buang terlalu pendek / terlalu panjang (listing nyangkut)
            continue
        rows.append({"id_artikel": row.id_artikel, "id_kalimat": f"{row.id_artikel}_{i}",
                     "bank": banks, "timestamp": row.timestamp, "kalimat": bersih, "n_kata": n})
kal = pd.DataFrame(rows)
total_sebelum_dedup = len(kal)

# --- dedup: kalimat identik yang berulang di banyak artikel disimpan 1x ---
kal = kal.drop_duplicates(subset="kalimat").reset_index(drop=True)
print(f"Kalimat setelah bersih : {total_sebelum_dedup:,}")
print(f"Duplikat dibuang       : {total_sebelum_dedup - len(kal):,}")
print(f"Kalimat unik (final)   : {len(kal):,}")

## **3. EDA ringkas**

Pemeriksaan cepat atas data clean (tidak mengubah data).

In [ ]:
print("Statistik jumlah kata per kalimat:")
print(kal["n_kata"].describe().round(1))
print("\nDistribusi panjang kalimat (kata):")
bins = [0, 5, 10, 15, 20, 30, 50, 81]
print(pd.cut(kal["n_kata"], bins=bins, right=False).value_counts().sort_index())
print("\nRentang tanggal:", raw.timestamp.min(), "-", raw.timestamp.max())
print("Jumlah bank yang disebut per kalimat:")
print(kal["bank"].map(lambda x: len(x.split(", ")) if x else 0).value_counts().sort_index())
print("Kalimat tanpa bank terdeteksi:", (kal["bank"] == "").sum())

## **4. Stopword untuk keyword topik**

Dipakai HANYA untuk representasi keyword c-TF-IDF, bukan teks yang di-embed. Final = (generik Sastrawi + domain finansial) DIKURANGI kata negasi (negasi menentukan arah sentimen).

In [ ]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

tokens = []
for k in kal["kalimat"]:
    tokens += re.findall(r"[a-zA-Z]+", str(k).lower())
freq = Counter(tokens)
print(f"Total token: {len(tokens):,} | kata unik: {len(freq):,}")

stop_generik = set(StopWordRemoverFactory().get_stop_words())
stop_domain = {"rp", "pt", "tbk", "persero", "miliar", "triliun", "juta", "ribu",
               "i", "ii", "iii", "yoy", "as", "us",
               "senin", "selasa", "rabu", "kamis", "jumat", "sabtu", "minggu",
               "indonesia", "menjadi", "ihsg"}
NEGASI = {"tidak", "tak", "bukan", "belum", "jangan", "tanpa", "kurang",
          "non", "anti", "minus", "gagal", "batal", "turun", "melemah", "koreksi"}
stopwords_id = sorted((stop_generik | stop_domain) - NEGASI)
assert not (set(stopwords_id) & NEGASI)
print(f"Stopword final untuk vectorizer: {len(stopwords_id)} (negasi dipertahankan)")

---
## **5. BERTopic + outlier handling**

Input = kalimat unik. `min_cluster_size=300`, `random_state=42` (reproducible). Setelah fit, **outlier (-1) dialihkan ke topik terdekat** dengan `reduce_outliers` (strategi embeddings) sehingga outlier berkurang; kalimat yang masih -1 benar-benar tak punya kecocokan.

In [ ]:
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic

SEED, MIN_CLUSTER_SIZE, MIN_SAMPLES, MIN_DF = 42, 300, 10, 5
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

docs = kal["kalimat"].tolist()
print(f"Input BERTopic: {len(docs):,} kalimat unik")

embedding_model = SentenceTransformer(MODEL_NAME)
embeddings = embedding_model.encode(docs, show_progress_bar=True, batch_size=64)

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=SEED)
hdbscan_model = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES,
                        metric="euclidean", cluster_selection_method="eom", prediction_data=True)
vectorizer_model = CountVectorizer(stop_words=stopwords_id, ngram_range=(1, 2), min_df=MIN_DF)

topic_model = BERTopic(embedding_model=embedding_model, umap_model=umap_model,
                       hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model,
                       language="multilingual", calculate_probabilities=False, verbose=True)
topics, probs = topic_model.fit_transform(docs, embeddings)

n_out_awal = int((pd.Series(topics) == -1).sum())
print(f"Sebelum reduce -> topik: {sum(topic_model.get_topic_info().Topic != -1)} | outlier: {n_out_awal:,} ({n_out_awal/len(topics):.1%})")

# outlier handling: alihkan -1 ke topik terdekat berdasarkan embeddings
topics = topic_model.reduce_outliers(docs, topics, strategy="embeddings", embeddings=embeddings)
topic_model.update_topics(docs, topics=topics, vectorizer_model=vectorizer_model)

info = topic_model.get_topic_info()
n_out = int((pd.Series(topics) == -1).sum())
print(f"Sesudah reduce -> topik: {sum(info.Topic != -1)} | outlier: {n_out:,} ({n_out/len(topics):.1%})")
info.head(30)

## **6. Simpan 2 output final**

In [ ]:
OUTDIR = Path("data/processed")
OUTDIR.mkdir(parents=True, exist_ok=True)

# OUTPUT 1: seluruh kalimat clean (tanpa duplikat) + topik
kal = kal.copy()
kal["topic"] = topics
kal[["id_artikel", "id_kalimat", "bank", "timestamp", "kalimat", "n_kata", "topic"]].to_csv(
    OUTDIR / "sentences_with_topic.csv", index=False)
print(f"[1] sentences_with_topic.csv -> {len(kal):,} kalimat")

# OUTPUT 2: info tiap topik + 3 kalimat representatif
baris = []
for t in info["Topic"].tolist():
    kw = ", ".join(w for w, _ in (topic_model.get_topic(t) or [])[:10])
    docs_rep = (topic_model.get_representative_docs(t) or []) if t != -1 else []
    docs_rep = (docs_rep + ["", "", ""])[:3]
    baris.append({"Topic": t, "Count": int(info.loc[info.Topic == t, "Count"].iloc[0]),
                  "Keywords": kw, "Representative_1": docs_rep[0],
                  "Representative_2": docs_rep[1], "Representative_3": docs_rep[2]})
pd.DataFrame(baris).to_csv(OUTDIR / "topic_info.csv", index=False)
print(f"[2] topic_info.csv -> {len(baris)} baris topik")

### **Langkah manual berikutnya**

Namai & gabungkan topik menjadi **10-15 aspek**, rujuk **FiQA Corporate & Stock**; topik tak jelas -> "Lainnya". Tulis di `daftar_aspek.md`, lalu **kunci daftar aspek** sebelum anotasi. Pemetaan topik->aspek dikerjakan manual.